# Automatic Batch Pipeline — ΔPE / JSD by Skill Tier (v4)

**No manual driver/stint selection anywhere in this notebook.** Every cell below reads
`datasets.py` directly and runs both required comparisons, for both tracks, end to end:

1. **Intermediate (test) × Experienced (reference)** → Tomaz vs Rodrigo
2. **Amateur (test) × Intermediate (reference)** → Morsinaldo vs Tomaz

For each (track × comparison) combination:
- The stint with the **most valid laps** is picked automatically per driver (documented,
  reproducible rule — Reviewer 1, point 3).
- **JSD** (fixed n = 20 bins) and **Permutation Entropy (PE, m = 3, τ = 1)** are computed
  per sector, for the 4 channels in `VARIABLES_ENTROPY`.
- ΔPE and JSD are correlated with Δ sector time (Pearson **and** Spearman).
- Statistical significance is tested with the **Wilcoxon signed-rank** test (paired by
  sector), corrected for multiple testing with **FDR (Benjamini–Hochberg)**.
- Every figure is saved **in English**, as PDF, under
  `BASE_IMG_DIR/batch_delta_pe_skill_tiers/<track>/<comparison>/`.
- Section 13 assembles **publication-ready summary figures** (combined grids across
  tracks and comparisons) for direct inclusion in the manuscript.

> **v4 change log:** raw / normalised Shannon entropy (H, H_norm) and the KL divergence
> were removed — the notebook now focuses purely on **PE** and **JSD**. The Mann-Whitney U
> test was dropped: the **Wilcoxon signed-rank** test is the single significance test.

Run all cells top to bottom — there is nothing to configure except the block below.


## 0. Imports

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import jensenshannon
from scipy.stats import pearsonr, spearmanr, wilcoxon, rankdata
from statsmodels.stats.multitest import multipletests

try:
    import antropy as ant
    ANTROPY_AVAILABLE = True
except ImportError:
    ANTROPY_AVAILABLE = False
    print("!  antropy not found - permutation entropy will be skipped.\n"
          "   Install with:  pip install antropy")

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from config import (TRACK_CONFIGS, DATASETS, DRIVER_ALIAS, resolve_stint_files,
                     load_stint, basic_clean_and_units, build_lap_validity_table,
                     assign_sectors)

load_from_ibt = load_stint

print(f"Tracks available : {list(TRACK_CONFIGS.keys())}")
print(f"Driver aliases   : {DRIVER_ALIAS}")
print(f"antropy          : {ANTROPY_AVAILABLE}")

## 1. Configuration — skill tiers, comparisons, tracks (fixed by design, nothing to pick)

In [ ]:
# ── Skill tiers & comparisons - fixed by study design, no manual selection ──

DRIVER_TIER = {
    'Rodrigo':    {'alias': DRIVER_ALIAS.get('Rodrigo', 'Driver A'),    'tier': 'Experienced'},
    'Tomaz':      {'alias': DRIVER_ALIAS.get('Tomaz', 'Driver B'),      'tier': 'Intermediate'},
    'Morsinaldo': {'alias': DRIVER_ALIAS.get('Morsinaldo', 'Driver C'), 'tier': 'Amateur'},
    'Thallys':    {'alias': DRIVER_ALIAS.get('Thallys', 'Driver D'),    'tier': 'Amateur'},
    'Igor':       {'alias': DRIVER_ALIAS.get('Igor', 'Driver E'),       'tier': 'Amateur'},
    'Hilton':     {'alias': DRIVER_ALIAS.get('Hilton', 'Driver F'),     'tier': 'Amateur'},
}

# ref = referencia de maior skill · test = piloto avaliado
# Sign convention (mantida da versao anterior): Delta = test - reference
#   Grupo 1: Driver B vs Driver A                         (ambas as pistas)
#   Grupo 2: cada amador (C, D, E, F) individualmente vs Driver B  (ambas as pistas)
SKILL_COMPARISONS = [
    {'id': 'B_vs_A_Tomaz_Rodrigo',    'ref': 'Rodrigo', 'test': 'Tomaz'},
    {'id': 'C_vs_B_Morsinaldo_Tomaz', 'ref': 'Tomaz',    'test': 'Morsinaldo'},
    {'id': 'D_vs_B_Thallys_Tomaz',    'ref': 'Tomaz',    'test': 'Thallys'},
    {'id': 'E_vs_B_Igor_Tomaz',       'ref': 'Tomaz',    'test': 'Igor'},
    {'id': 'F_vs_B_Hilton_Tomaz',     'ref': 'Tomaz',    'test': 'Hilton'},
]

# Grupo usado nas celulas de insight da Secao 13 (concordancia de sinal)
COMPARISON_GROUP = {
    'B_vs_A_Tomaz_Rodrigo':    'Grupo 1: Driver B vs Driver A',
    'C_vs_B_Morsinaldo_Tomaz': 'Grupo 2: Amadores (C,D,E,F) vs Driver B',
    'D_vs_B_Thallys_Tomaz':    'Grupo 2: Amadores (C,D,E,F) vs Driver B',
    'E_vs_B_Igor_Tomaz':       'Grupo 2: Amadores (C,D,E,F) vs Driver B',
    'F_vs_B_Hilton_Tomaz':     'Grupo 2: Amadores (C,D,E,F) vs Driver B',
}

BATCH_TRACKS = ['charlotte_roval_2025', 'summit_point']   # both tracks, as scoped

MIN_VALID_LAPS_STINT = 3     # minimum valid laps to accept a stint as usable
FDR_ALPHA             = 0.05  # nominal alpha, before/after Benjamini-Hochberg correction

PROJECT_ROOT_IMG = Path.home() / "OneDrive/Documents/GitHub/Doutorado/Racing4all"
BASE_IMG_DIR      = PROJECT_ROOT_IMG / "Iracing" / "img"
BATCH_SAVE_ROOT   = BASE_IMG_DIR / "batch_delta_pe_skill_tiers"
BATCH_SAVE_ROOT.mkdir(parents=True, exist_ok=True)

print("Nothing to configure below - drivers, tracks and comparisons are fixed above.")
print(f"Comparisons : {[c['id'] for c in SKILL_COMPARISONS]}")
print(f"Tracks      : {BATCH_TRACKS}")
print(f"Output root : {BATCH_SAVE_ROOT}")

## 2. Divergence utilities — JSD with fixed binning (n = 20)


**Binning (v3, unchanged in v4): fixed `n_bins = 20`.** Histograms feeding the JSD use a
**fixed bin count of 20** for every driver, sector and channel, instead of the previous
adaptive Freedman-Diaconis rule. The bin count is therefore a single, reportable parameter,
and JSD values are directly comparable between drivers, sectors and tracks (Reviewer 1,
point 3). `freedman_diaconis_bins()` is kept in the cell below only as an optional
post-hoc sensitivity check, not as the method used to produce the reported results.

**v4:** Shannon entropy (H / H_norm) and KL divergence utilities were removed — the
information-theoretic core of the notebook is now **JSD + PE** only.


In [ ]:
# ── v4  Fixed binning (n = 20) + JSD utility ──────────────────────────────────────
# n_bins fixed at 20 for every driver / sector / channel JSD comparison, so results
# are directly comparable across drivers, sectors and tracks. Freedman-Diaconis is kept below only as an optional post-hoc
# sensitivity check - NOT used to generate the results reported in the manuscript.

N_BINS_FIXED = 20   # primary, non-adaptive bin count (Reviewer 1, point 3)


def freedman_diaconis_bins(data: np.ndarray, v_min: float, v_max: float,
                            min_bins: int = 5, max_bins: int = 100) -> int:
    """
    Freedman-Diaconis rule: bin_width = 2 * IQR * n^(-1/3)
    Adapts to sample size and data spread; falls back to sqrt(n) if IQR = 0.
    Kept for post-hoc sensitivity analysis only - not used by default below.
    """
    data = data[np.isfinite(data)]
    n    = len(data)
    if n < 4:
        return min_bins
    iqr = np.percentile(data, 75) - np.percentile(data, 25)
    if iqr > 0:
        bin_width = 2.0 * iqr * (n ** (-1.0 / 3.0))
        n_bins    = int(np.ceil((v_max - v_min) / bin_width))
    else:
        n_bins = max(min_bins, int(np.ceil(np.sqrt(n))))
    return int(np.clip(n_bins, min_bins, max_bins))


def pmf(data: np.ndarray, n_bins: int, v_range: tuple,
        smoothing: float = 1e-8) -> np.ndarray:
    """Probability Mass Function via histogram (density=False, then normalised)."""
    counts, _ = np.histogram(data[np.isfinite(data)], bins=n_bins, range=v_range)
    counts    = counts.astype(float) + smoothing
    return counts / counts.sum()


def compute_jsd(data_a: np.ndarray, data_b: np.ndarray, v_range: tuple,
                n_bins: int = N_BINS_FIXED) -> tuple:
    """Jensen-Shannon Divergence in [0, 1] (log base-2 / bits). Returns (jsd, n_bins).
    n_bins is fixed at 20 by default; pass a different value only for sensitivity checks."""
    p     = pmf(data_a, n_bins, v_range)
    q     = pmf(data_b, n_bins, v_range)
    jsd_d = float(jensenshannon(p, q, base=2) ** 2)
    return jsd_d, n_bins


print(f"JSD utilities loaded. Fixed binning: n_bins = {N_BINS_FIXED} "
      f"(H_max = {np.log2(N_BINS_FIXED):.3f} bits)")


## 3. Variables and global ranges (computed once, per track, across all recorded drivers)

In [ ]:
VARIABLES_ENTROPY = [
    ('TotalAccel_G',       'Combined G-Force'),
    ('Throttle_Pct',       'Throttle (%)'),
    ('Brake_Pct',          'Brake (%)'),
    ('SteeringWheelAngle', 'Steering Angle (deg)'),
]

def compute_global_ranges(datasets: dict, variables: list,
                           track_filter: str = None,
                           lower_pct: float = 1,
                           upper_pct: float = 99) -> dict:
    col_values = {col: [] for col, _ in variables}
    for track_key, track_info in datasets.items():
        if track_filter and track_key != track_filter:
            continue
        base_path = Path(track_info["base_path"])
        for pilot, stints in track_info["sessions"].items():
            for stint_key, filenames in stints.items():
                if isinstance(filenames, str):
                    filenames = [filenames]
                filepaths = [base_path / f for f in filenames]
                try:
                    df_raw = load_from_ibt(filepaths)
                    if df_raw.empty:
                        continue
                    df_tmp = basic_clean_and_units(df_raw)
                    if 'TotalAccel_G' not in df_tmp.columns:
                        df_tmp['TotalAccel_G'] = np.sqrt(
                            (df_tmp['LatAccel'] / 9.81) ** 2 + (df_tmp['LongAccel'] / 9.81) ** 2)
                    for col, _ in variables:
                        if col in df_tmp.columns:
                            col_values[col].append(df_tmp[col].dropna().values)
                except Exception as e:
                    print(f"  !  {track_key} | {pilot} | {stint_key} - {e}")

    global_ranges = {}
    for col, _ in variables:
        if col_values[col]:
            all_vals = np.concatenate(col_values[col])
            v_min, v_max = np.percentile(all_vals, [lower_pct, upper_pct])
            global_ranges[col] = (v_min, v_max)
        else:
            print(f"  !  No data for '{col}'")
    return global_ranges

print("Global-range function loaded.")

## 4. Shared helpers: lap-distance alignment and lap preprocessing

In [ ]:
from typing import Dict

def align_lap_by_dist(g: pd.DataFrame, grid: np.ndarray) -> Dict[str, np.ndarray]:
    g = g.sort_values("LapDistPct").drop_duplicates(subset=["LapDistPct"], keep="first")
    if g.empty:
        return {}
    t_rel = g["SessionTime"] - g["SessionTime"].iloc[0]
    x     = g["LapDistPct"].to_numpy()
    if len(x) < 2 or np.allclose(x.max() - x.min(), 0):
        return {}
    def interp(y): return np.interp(grid, x, y)
    return {
        "LapDistPct":         grid,
        "t_rel":              interp(t_rel.to_numpy()),
        "speed":              interp(g["Speed_KPH"].to_numpy()),
        "throttle":           interp(g["Throttle_Pct"].to_numpy()),
        "brake":              interp(g["Brake_Pct"].to_numpy()),
        "SteeringWheelAngle": interp(g["SteeringWheelAngle"].to_numpy()),
        "YawRate":            interp(g.get("YawRate",   pd.Series(np.zeros_like(x))).to_numpy()),
        "LongAccel":          interp(g.get("LongAccel", pd.Series(np.zeros_like(x))).to_numpy()),
    }


def preprocess_ibt_dataframe(df):
    """Adds LocalLapTime and TotalAccel_G columns."""
    df = df.copy()
    df.sort_values(['Lap', 'SessionTime'], inplace=True)
    df['LocalLapTime'] = df.groupby('Lap')['SessionTime'].transform(lambda x: x - x.min())
    if 'TotalAccel_G' not in df.columns:
        lat_g = df.get('LatAccel', pd.Series(0, index=df.index)) / 9.81
        lon_g = df.get('LongAccel', pd.Series(0, index=df.index)) / 9.81
        df['TotalAccel_G'] = np.sqrt(lat_g**2 + lon_g**2)
    return df

print("Shared helpers loaded.")

## 5. Automatic best-stint selection and batch loading (all drivers, both tracks, no manual picks)

In [ ]:
def _is_warmup(stint_name: str) -> bool:
    s = stint_name.lower()
    return s.startswith('warmup') or 'aqueciment' in s


def select_best_stint(track_id: str, driver: str, min_valid_laps: int = MIN_VALID_LAPS_STINT):
    """
    Loads every non-warmup stint of a driver on a track and returns the one
    with the most valid laps. Documented, reproducible selection rule
    (Reviewer 1, point 3) - no manual stint choice anywhere in this notebook.
    """
    cfg = DATASETS[track_id]
    stints = cfg['sessions'].get(driver, {})
    candidates = [s for s in stints if not _is_warmup(s)]

    best = None
    for stint_name in candidates:
        try:
            paths  = [str(p) for p in resolve_stint_files(track_id, driver, stint_name)]
            df_raw = load_from_ibt(paths)
            if df_raw.empty:
                continue
            df_cln   = basic_clean_and_units(df_raw)
            validity = build_lap_validity_table(df_cln, manual_invalid=set(), verbose=False)
            n_valid  = int(validity['Valid'].sum())
            print(f"    {driver:<12} {stint_name:<10} -> {n_valid} valid laps")
            if n_valid >= min_valid_laps and (best is None or n_valid > best['n_valid']):
                valid_laps = validity.loc[validity['Valid'], 'Lap'].tolist()
                df_clean   = df_cln[df_cln['Lap'].isin(valid_laps)].copy()
                df_clean   = preprocess_ibt_dataframe(df_clean)
                best = {'stint': stint_name, 'df': df_clean, 'n_valid': n_valid}
        except Exception as e:
            print(f"    !  {driver} {stint_name}: {e}")

    if best is None:
        print(f"    x  No valid stint for {driver} on {track_id}")
    else:
        print(f"    OK Selected: {driver} -> {best['stint']} ({best['n_valid']} laps)")
    return best


print("Loading every dataset for the 3 study drivers, both tracks (fully automatic)...\n")

BATCH_DATA = {}   # BATCH_DATA[track_id][driver] = {'stint':..., 'df':..., 'n_valid':...}

for track_id in BATCH_TRACKS:
    print(f"=== Track: {track_id} ===")
    BATCH_DATA[track_id] = {}
    for driver in DRIVER_TIER:
        if driver not in DATASETS[track_id]['sessions']:
            print(f"    !  {driver}: no data on {track_id}")
            continue
        result = select_best_stint(track_id, driver)
        if result is not None:
            BATCH_DATA[track_id][driver] = result
    print()

print("Batch loading complete - no manual selection was needed.")

## 6. Sector assignment and per-track global ranges

In [ ]:
BATCH_SECTORS = {}   # BATCH_SECTORS[track_id] = {'edges':..., 'names':..., 'track_name':...}
BATCH_RANGES  = {}   # BATCH_RANGES[track_id]  = global_ranges dict (for JSD)

for track_id in BATCH_TRACKS:
    cfg = TRACK_CONFIGS[track_id]
    BATCH_SECTORS[track_id] = {
        'edges': cfg['custom_edges'],
        'names': cfg['sector_names'],
        'track_name': cfg['track_name'],
    }

    print(f"Global ranges - {cfg['track_name']}")
    BATCH_RANGES[track_id] = compute_global_ranges(DATASETS, VARIABLES_ENTROPY, track_filter=track_id)

    edges = BATCH_SECTORS[track_id]['edges']
    for driver, rec in BATCH_DATA[track_id].items():
        rec['df_s'] = assign_sectors(rec['df'].copy(), edges)
    print()

## 7. Helper functions: sector-level PE, sector Δ time, Wilcoxon significance test


In [ ]:
def compute_pe_by_sector(df, target_col, sectors, sector_col,
                          order=3, delay=1, min_samples=5):
    """Normalised Permutation Entropy per sector (Bandt & Pompe, 2002),
    aggregated as mean/std over the per-lap PE values (m = order = 3, tau = delay = 1)."""
    results = {}
    for sector in sectors:
        sector_df = df[df[sector_col] == sector].sort_values(['Lap', 'SessionTime'])
        lap_pe = []
        for lap in sector_df['Lap'].unique():
            series = sector_df[sector_df['Lap'] == lap][target_col].dropna().values
            if len(series) >= min_samples:
                pe_val = ant.perm_entropy(series, order=order, delay=delay, normalize=True)
                lap_pe.append(float(pe_val))
        if lap_pe:
            results[sector] = {'pe_mean': float(np.mean(lap_pe)),
                                'pe_std': float(np.std(lap_pe)), 'n_laps': len(lap_pe)}
        else:
            results[sector] = {'pe_mean': np.nan, 'pe_std': np.nan, 'n_laps': 0}
    return results


def sector_time_delta(df_ref, df_test, custom_edges, sectors):
    """Delta time per sector from each driver's best valid lap."""
    grid = np.linspace(0.0, 1.0, 2000)

    def _best_lap_interp(df):
        validity = build_lap_validity_table(df, manual_invalid=set(), verbose=False)
        best_row = validity[validity['Valid']].sort_values('LapTime_s').iloc[0]
        g = df[df['Lap'] == int(best_row['Lap'])]
        return align_lap_by_dist(g, grid)

    ir = _best_lap_interp(df_ref)
    it = _best_lap_interp(df_test)
    if not ir or not it:
        return pd.DataFrame(columns=['Sector', 'DeltaTime_s'])

    loss     = it['t_rel'] - ir['t_rel']
    lap_dist = ir['LapDistPct']

    rows = []
    for s_id in sorted(sectors):
        if s_id > len(custom_edges) - 1:
            continue
        s0, s1 = custom_edges[s_id - 1], custom_edges[s_id]
        mask = (lap_dist >= s0) & (lap_dist <= s1)
        if not np.any(mask):
            continue
        pts = np.flatnonzero(mask)
        rows.append({'Sector': s_id, 'DeltaTime_s': float(loss[pts[-1]] - loss[pts[0]])})
    return pd.DataFrame(rows)


def wilcoxon_matched_by_sector(delta_values: np.ndarray):
    """Paired Wilcoxon signed-rank across sectors. H0: median(delta) = 0.
    Returns (stat, p, matched-pairs rank-biserial r, n)."""
    d = np.asarray(delta_values, dtype=float)
    d = d[np.isfinite(d)]
    d = d[d != 0]
    n = len(d)
    if n < 4:
        return np.nan, np.nan, np.nan, n
    stat, p = wilcoxon(d, alternative='two-sided', zero_method='wilcox')
    ranks   = rankdata(np.abs(d))
    w_pos   = ranks[d > 0].sum()
    w_neg   = ranks[d < 0].sum()
    r_rb    = (w_pos - w_neg) / (w_pos + w_neg)
    return float(stat), float(p), float(r_rb), n


def apply_fdr(df, pcol='p-value', alpha=FDR_ALPHA):
    """Adds 'p-adj (FDR)' and 'sig (FDR)' columns via Benjamini-Hochberg."""
    df = df.copy()
    mask = df[pcol].notna()
    df['p-adj (FDR)'] = np.nan
    df['sig (FDR)']   = False
    if mask.sum() > 0:
        rej, p_adj, _, _ = multipletests(df.loc[mask, pcol], alpha=alpha, method='fdr_bh')
        df.loc[mask, 'p-adj (FDR)'] = p_adj
        df.loc[mask, 'sig (FDR)']   = rej
    return df

print("Section 7 helper functions loaded.")

## 8. Plotting functions (all labels in English)

In [ ]:
def _plot_pe_heatmaps(df_pe_sector, sectors, label_ref, label_test, track_name,
                       save_dir, comp_id, track_id):
    if df_pe_sector.empty:
        print("  !  No PE data to plot."); return
    pivot_ref  = df_pe_sector.pivot(index='Sector', columns='Variable', values='PE_ref').reindex(sectors)
    pivot_test = df_pe_sector.pivot(index='Sector', columns='Variable', values='PE_test').reindex(sectors)

    n_vars = pivot_ref.shape[1]
    w = max(12, n_vars * 2.5); h = max(6, len(sectors) * 0.55)

    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(w * 2, h), gridspec_kw={'wspace': 0.08})
    hm_kw = dict(cmap='YlOrRd', vmin=0, vmax=1, annot=True, fmt='.3f',
                 annot_kws={'size': 9}, linewidths=0.4, linecolor='#dddddd',
                 cbar_kws={'label': 'PE_norm  [0 = ordered, 1 = random]', 'shrink': 0.7})
    sns.heatmap(pivot_ref,  ax=ax_l, mask=pivot_ref.isna(),  **hm_kw)
    sns.heatmap(pivot_test, ax=ax_r, mask=pivot_test.isna(), **hm_kw)
    ax_l.set_title(label_ref,  fontsize=13, fontweight='bold')
    ax_r.set_title(label_test, fontsize=13, fontweight='bold')
    for ax in (ax_l, ax_r):
        ax.set_xlabel('Variable')
        ax.tick_params(axis='x', labelsize=9, rotation=20)
        ax.tick_params(axis='y', labelsize=8)
    ax_l.set_ylabel('Sector (track order)')
    ax_r.set_ylabel('')
    fig.suptitle(f"Permutation Entropy by Sector - {track_name}\n"
                 f"{label_test} vs {label_ref}  (order=3, delay=1)",
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(save_dir / f"pe_heatmap_{comp_id}_{track_id}.pdf", dpi=300, bbox_inches='tight')
    plt.show()


def _plot_delta_pe_heatmap(df_pe_sector, sectors, label_ref, label_test, track_name,
                            save_dir, comp_id, track_id):
    if df_pe_sector.empty:
        return
    pivot = df_pe_sector.pivot(index='Sector', columns='Variable', values='DeltaPE').reindex(sectors)
    vabs  = max(0.05, np.nanmax(np.abs(pivot.values)) * 1.1) if pivot.notna().any().any() else 0.2

    fig, ax = plt.subplots(figsize=(max(10, pivot.shape[1] * 2.2), max(6, len(sectors) * 0.6)))
    sns.heatmap(pivot, ax=ax, cmap='RdBu_r', vmin=-vabs, vmax=vabs, center=0,
                annot=True, fmt='.3f', annot_kws={'size': 9},
                linewidths=0.4, linecolor='#cccccc', mask=pivot.isna(),
                cbar_kws={'label': 'DeltaPE (test - ref)  [red = more random, blue = more ordered]',
                          'shrink': 0.7})
    ax.set_title(f"DeltaPE by Sector - {label_test} minus {label_ref}\n{track_name}",
                 fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Variable'); ax.set_ylabel('Sector (track order)')
    ax.tick_params(axis='x', labelsize=9, rotation=20)
    ax.tick_params(axis='y', labelsize=8)
    plt.tight_layout()
    fig.savefig(save_dir / f"delta_pe_heatmap_{comp_id}_{track_id}.pdf", dpi=300, bbox_inches='tight')
    plt.show()


def _plot_correlation_summary(df_corr, save_dir, comp_id, track_id, track_name, label_ref, label_test):
    if df_corr.empty:
        print("  !  No correlation data to plot."); return
    fig, ax = plt.subplots(figsize=(8, max(4, len(df_corr) * 0.4)))
    labels = df_corr['Metric'] + ' - ' + df_corr['Variable']
    y_pos  = np.arange(len(df_corr))
    colors = ['firebrick' if s else 'steelblue' for s in df_corr['Pearson_sig_FDR']]
    ax.barh(y_pos, df_corr['Pearson_r'], color=colors, alpha=0.85, edgecolor='white')
    ax.set_yticks(y_pos); ax.set_yticklabels(labels, fontsize=9)
    ax.axvline(0, color='gray', lw=1)
    ax.set_xlabel('Pearson r  (metric vs Delta sector time)')
    ax.set_title(f"Correlation with Delta Sector Time - {label_test} vs {label_ref}\n"
                 f"{track_name}   (red = significant after FDR correction, q<0.05)",
                 fontsize=12, fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_dir / f"correlation_summary_{comp_id}_{track_id}.pdf", dpi=300, bbox_inches='tight')
    plt.show()


def _plot_significance_summary(df_sig, save_dir, comp_id, track_id, track_name):
    if df_sig.empty:
        print("  !  No significance data to plot."); return
    fig, ax = plt.subplots(figsize=(9, max(4, len(df_sig) * 0.35)))
    test_short = df_sig['Test'].str.replace(r' \(.*\)', '', regex=True)
    labels = test_short + ' - ' + df_sig['Metric'] + ' - ' + df_sig['Variable']
    y_pos  = np.arange(len(df_sig))
    colors = ['darkgreen' if s else 'lightgray' for s in df_sig['sig (FDR)']]
    ax.barh(y_pos, -np.log10(df_sig['p-value'].clip(lower=1e-10)), color=colors, edgecolor='white')
    ax.axvline(-np.log10(0.05), color='firebrick', ls='--', lw=1.2, label='p = 0.05')
    ax.set_yticks(y_pos); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('-log10(p-value)')
    ax.set_title(f"Statistical Significance Summary - {track_name}\n"
                 f"(green = significant after FDR / Benjamini-Hochberg correction)",
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_dir / f"significance_summary_{comp_id}_{track_id}.pdf", dpi=300, bbox_inches='tight')
    plt.show()

print("Plotting functions loaded (English labels).")

## 9. Main pipeline function: JSD + PE + correlation + Wilcoxon significance, per track × comparison


In [ ]:
ALL_RESULTS = []   # collected dicts for the final combined summary

def run_skill_tier_analysis(track_id, comparison):
    ref_driver, test_driver = comparison['ref'], comparison['test']
    comp_id = comparison['id']

    if ref_driver not in BATCH_DATA[track_id] or test_driver not in BATCH_DATA[track_id]:
        print(f"  !  Insufficient data for {comp_id} on {track_id} - skipping.")
        return None

    df_ref_s  = BATCH_DATA[track_id][ref_driver]['df_s']
    df_test_s = BATCH_DATA[track_id][test_driver]['df_s']
    df_ref    = BATCH_DATA[track_id][ref_driver]['df']
    df_test   = BATCH_DATA[track_id][test_driver]['df']

    edges      = BATCH_SECTORS[track_id]['edges']
    names      = BATCH_SECTORS[track_id]['names']
    track_name = BATCH_SECTORS[track_id]['track_name']
    ranges     = BATCH_RANGES[track_id]

    sector_col = 'Sector' if 'Sector' in df_ref_s.columns else 'sector'
    sectors_present = [s for s in names
                       if s in df_ref_s[sector_col].unique() and s in df_test_s[sector_col].unique()]

    label_ref  = f"{DRIVER_TIER[ref_driver]['alias']} ({DRIVER_TIER[ref_driver]['tier']})"
    label_test = f"{DRIVER_TIER[test_driver]['alias']} ({DRIVER_TIER[test_driver]['tier']})"

    save_dir = BATCH_SAVE_ROOT / track_id / comp_id
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*90}\n  {comp_id.upper()}  |  {track_name}\n"
          f"  Reference: {label_ref}   |   Test: {label_test}\n{'='*90}")

    # ── (a) JSD per sector ──────────────────────────────────────────────────
    rows_jsd = []
    for col, label in VARIABLES_ENTROPY:
        v_range = ranges.get(col)
        if v_range is None or col not in df_ref_s.columns:
            continue
        for sector in sectors_present:
            a = df_ref_s[df_ref_s[sector_col] == sector][col].dropna().values
            b = df_test_s[df_test_s[sector_col] == sector][col].dropna().values
            if len(a) < 10 or len(b) < 10:
                continue
            jsd_v, _ = compute_jsd(a, b, v_range)
            rows_jsd.append({'Sector': sector, 'Variable': label, 'JSD': jsd_v})
    df_jsd_sector = pd.DataFrame(rows_jsd)

    # ── (b) PE per sector ───────────────────────────────────────────────────
    rows_pe = []
    for col, label in VARIABLES_ENTROPY:
        if col not in df_ref_s.columns:
            continue
        pe_ref  = compute_pe_by_sector(df_ref_s,  col, sectors_present, sector_col)
        pe_test = compute_pe_by_sector(df_test_s, col, sectors_present, sector_col)
        for sector in sectors_present:
            r, t = pe_ref[sector], pe_test[sector]
            if np.isnan(r['pe_mean']) or np.isnan(t['pe_mean']):
                continue
            rows_pe.append({'Sector': sector, 'Variable': label,
                             'PE_ref': r['pe_mean'], 'PE_test': t['pe_mean'],
                             'DeltaPE': t['pe_mean'] - r['pe_mean'],
                             'n_laps_ref': r['n_laps'], 'n_laps_test': t['n_laps']})
    df_pe_sector = pd.DataFrame(rows_pe)

    # ── (c) Delta time per sector (best lap) ────────────────────────────────
    df_delta = sector_time_delta(df_ref, df_test, edges, sectors_present)

    # ── (d) Correlation: DeltaPE / JSD  vs  Delta sector time ───────────────
    corr_rows = []
    for metric_col, df_metric in [('DeltaPE', df_pe_sector),
                                   ('JSD', df_jsd_sector)]:
        if df_metric.empty or df_delta.empty:
            continue
        pivot  = df_metric.pivot(index='Sector', columns='Variable', values=metric_col)
        merged = pivot.merge(df_delta.set_index('Sector'), left_index=True, right_index=True).dropna()
        for var_label in pivot.columns:
            x = merged[var_label].values
            y = merged['DeltaTime_s'].values
            if len(x) < 4:
                continue
            r_p, p_p = pearsonr(x, y)
            r_s, p_s = spearmanr(x, y)
            corr_rows.append({'Metric': metric_col, 'Variable': var_label, 'n_sectors': len(x),
                               'Pearson_r': r_p, 'Pearson_p': p_p,
                               'Spearman_rho': r_s, 'Spearman_p': p_s})
    df_corr = pd.DataFrame(corr_rows)
    if not df_corr.empty:
        df_corr = apply_fdr(df_corr, pcol='Pearson_p')
        df_corr.rename(columns={'p-adj (FDR)': 'Pearson_p_FDR', 'sig (FDR)': 'Pearson_sig_FDR'},
                        inplace=True)

    # ── (e) Significance: Wilcoxon signed-rank (paired by sector) on DeltaPE ─
    sig_rows = []
    for var_label in [l for _, l in VARIABLES_ENTROPY]:
        if df_pe_sector.empty:
            continue
        vals = df_pe_sector.loc[df_pe_sector['Variable'] == var_label, 'DeltaPE'].dropna().values
        stat, p, r_rb, n = wilcoxon_matched_by_sector(vals)
        sig_rows.append({'Test': 'Wilcoxon (paired, by sector)', 'Metric': 'DeltaPE',
                          'Variable': var_label, 'stat': stat, 'p-value': p,
                          'effect_r': r_rb, 'n': n})

    df_sig = pd.DataFrame(sig_rows)
    if not df_sig.empty:
        df_sig = apply_fdr(df_sig, pcol='p-value')

    # ── (f) Figures (all labels in English) ─────────────────────────────────
    _plot_pe_heatmaps(df_pe_sector, sectors_present, label_ref, label_test, track_name,
                       save_dir, comp_id, track_id)
    _plot_delta_pe_heatmap(df_pe_sector, sectors_present, label_ref, label_test, track_name,
                            save_dir, comp_id, track_id)
    _plot_correlation_summary(df_corr, save_dir, comp_id, track_id, track_name, label_ref, label_test)
    _plot_significance_summary(df_sig, save_dir, comp_id, track_id, track_name)

    # ── (g) Save tables ──────────────────────────────────────────────────────
    df_pe_sector.to_csv(save_dir / f"pe_by_sector_{comp_id}_{track_id}.csv", index=False)
    df_jsd_sector.to_csv(save_dir / f"jsd_by_sector_{comp_id}_{track_id}.csv", index=False)
    df_corr.to_csv(save_dir / f"correlation_summary_{comp_id}_{track_id}.csv", index=False)
    df_sig.to_csv(save_dir / f"significance_tests_{comp_id}_{track_id}.csv", index=False)

    print(f"  Figures and tables saved to: {save_dir}")

    return {'track_id': track_id, 'comparison': comp_id, 'track_name': track_name,
            'label_ref': label_ref, 'label_test': label_test,
            'df_pe_sector': df_pe_sector, 'df_jsd_sector': df_jsd_sector,
            'df_corr': df_corr, 'df_sig': df_sig, 'df_delta': df_delta,
            'n_sectors': len(sectors_present)}

print("Main pipeline function loaded.")

## 10. Run everything — both tracks x both comparisons, automatically

In [ ]:
for track_id in BATCH_TRACKS:
    for comparison in SKILL_COMPARISONS:
        result = run_skill_tier_analysis(track_id, comparison)
        if result is not None:
            ALL_RESULTS.append(result)

print(f"\n{len(ALL_RESULTS)} track x comparison combinations processed automatically.")

## 11. Combined summary tables (for Methods / Supplementary Material)

In [ ]:
combined_sig_rows = []
for res in ALL_RESULTS:
    if res['df_sig'].empty:
        continue
    df = res['df_sig'].copy()
    df.insert(0, 'Comparison', res['comparison'])
    df.insert(0, 'Track', res['track_name'])
    combined_sig_rows.append(df)

df_sig_all = pd.concat(combined_sig_rows, ignore_index=True) if combined_sig_rows else pd.DataFrame()

combined_corr_rows = []
for res in ALL_RESULTS:
    if res['df_corr'].empty:
        continue
    df = res['df_corr'].copy()
    df.insert(0, 'Comparison', res['comparison'])
    df.insert(0, 'Track', res['track_name'])
    combined_corr_rows.append(df)

df_corr_all = pd.concat(combined_corr_rows, ignore_index=True) if combined_corr_rows else pd.DataFrame()

print("=" * 90)
print("  COMBINED STATISTICAL SIGNIFICANCE TABLE - ALL TRACKS x COMPARISONS")
print("=" * 90)
display(df_sig_all)

print("\n" + "=" * 90)
print("  COMBINED CORRELATION TABLE - DeltaPE / JSD vs DELTA SECTOR TIME")
print("=" * 90)
display(df_corr_all)

csv_sig  = BATCH_SAVE_ROOT / "combined_significance_tests_all.csv"
csv_corr = BATCH_SAVE_ROOT / "combined_correlation_summary_all.csv"
df_sig_all.to_csv(csv_sig, index=False)
df_corr_all.to_csv(csv_corr, index=False)
print(f"\nCombined tables saved:\n  {csv_sig}\n  {csv_corr}")

## 12. Interpretation notes and limitations (for Methods / Discussion)

- **Sector counts are small** (10 for Summit Point, 18 for Charlotte Roval), so the paired
  Wilcoxon signed-rank test should be read as *exploratory*, even after FDR correction —
  report effect sizes (`effect_r`, matched-pairs rank-biserial) alongside p-values in the
  manuscript, not p-values alone.
- **Binning is fixed at n = 20** for the JSD (Section 2), replacing the earlier adaptive
  Freedman-Diaconis rule, so the bin count is a single reportable parameter and JSD values
  are comparable across drivers/sectors/tracks (Reviewer 1, point 3). Freedman-Diaconis
  remains available in the code only as an optional post-hoc sensitivity check.
- **FDR (Benjamini-Hochberg)** was chosen over Bonferroni because Bonferroni's full-family
  correction is overly conservative for N as small as ours, inflating the false-negative
  rate; FDR controls the *expected proportion* of false positives among significant results,
  keeping more statistical power — a standard choice for small-N exploratory studies with
  many correlated tests (adjacent sectors are not independent).
- **v4 scope:** raw / normalised Shannon entropy, KL divergence and the Mann-Whitney U test
  were removed. The information-theoretic core is **PE (m = 3, τ = 1)** for temporal
  structure and **JSD** for distributional divergence; the **Wilcoxon signed-rank** test
  (paired by sector) is the single significance test, keeping the statistical narrative
  minimal and consistent with the manuscript.


## 13. Publication-ready summary figures (for the manuscript)

The cells below assemble **combined figures across both tracks and both skill-tier
comparisons**, ready for direct inclusion in the article. All figures are saved as
vector PDF (embedded TrueType fonts, `fonttype = 42`) under
`BATCH_SAVE_ROOT/article_figures/`:

- **Fig. A — `fig_delta_pe_grid.pdf`:** ΔPE (test − reference) heatmaps, one panel per
  track × comparison. Red = the lower-skill driver is *more random*; blue = *more ordered*.
- **Fig. B — `fig_jsd_grid.pdf`:** JSD heatmaps (fixed n = 20 bins), same panel layout.
- **Fig. C — `fig_wilcoxon_summary.pdf`:** Wilcoxon signed-rank effect sizes
  (matched-pairs rank-biserial r) per channel, with FDR-significant results highlighted.
- **Fig. D — `fig_deltape_vs_deltatime.pdf`:** ΔPE vs Δ sector time scatter, one panel
  per channel, pooling all tracks/comparisons, with pooled Spearman ρ annotated.


In [ ]:
# ── 13.0  Publication style + output directory ─────────────────────────────────
import matplotlib as mpl

ARTICLE_FIG_DIR = BATCH_SAVE_ROOT / "article_figures"
ARTICLE_FIG_DIR.mkdir(parents=True, exist_ok=True)

PUB_RC = {
    'pdf.fonttype': 42, 'ps.fonttype': 42,          # editable text in the PDF
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'figure.dpi': 120, 'savefig.dpi': 300,
}

VAR_LABELS = [label for _, label in VARIABLES_ENTROPY]

def _panel_title(res):
    return f"{res['track_name']}\n{res['label_test']} vs {res['label_ref']}"

def _save(fig, name):
    path = ARTICLE_FIG_DIR / name
    fig.savefig(path, bbox_inches='tight')
    print(f"  saved: {path}")

print(f"Article figures will be saved to: {ARTICLE_FIG_DIR}")
print(f"{len(ALL_RESULTS)} track x comparison results available.")


In [ ]:
# ── 13.1  Fig. A — Combined DeltaPE heatmap grid (all tracks x comparisons) ─────
with mpl.rc_context(PUB_RC):
    n_panels = len(ALL_RESULTS)
    max_sectors = max(res['n_sectors'] for res in ALL_RESULTS)
    fig, axes = plt.subplots(1, n_panels,
                             figsize=(4.2 * n_panels, max(4.5, 0.42 * max_sectors)),
                             sharey=False)
    axes = np.atleast_1d(axes)

    vabs = max(0.05, max(np.nanmax(np.abs(res['df_pe_sector']['DeltaPE'].values))
                          for res in ALL_RESULTS if not res['df_pe_sector'].empty) * 1.05)

    for k, (ax, res) in enumerate(zip(axes, ALL_RESULTS)):
        piv = (res['df_pe_sector']
               .pivot(index='Sector', columns='Variable', values='DeltaPE')
               .reindex(columns=VAR_LABELS))
        sns.heatmap(piv, ax=ax, cmap='RdBu_r', vmin=-vabs, vmax=vabs, center=0,
                    annot=True, fmt='.2f', annot_kws={'size': 7},
                    linewidths=0.4, linecolor='#cccccc', mask=piv.isna(),
                    cbar=(k == n_panels - 1),
                    cbar_kws={'label': r'$\Delta$PE (test $-$ reference)', 'shrink': 0.8})
        ax.set_title(_panel_title(res), fontweight='bold')
        ax.set_xlabel(''); ax.set_ylabel('Sector' if k == 0 else '')
        ax.tick_params(axis='x', rotation=30)

    fig.suptitle(r'Permutation Entropy differences by sector ($m=3$, $\tau=1$; '
                 r'red = more random, blue = more ordered)',
                 fontweight='bold', y=1.04)
    plt.tight_layout()
    _save(fig, 'fig_delta_pe_grid.pdf')
    plt.show()


In [ ]:
# ── 13.2  Fig. B — Combined JSD heatmap grid (fixed n = 20 bins) ────────────────
with mpl.rc_context(PUB_RC):
    n_panels = len(ALL_RESULTS)
    max_sectors = max(res['n_sectors'] for res in ALL_RESULTS)
    fig, axes = plt.subplots(1, n_panels,
                             figsize=(4.2 * n_panels, max(4.5, 0.42 * max_sectors)))
    axes = np.atleast_1d(axes)

    jsd_max = max(np.nanmax(res['df_jsd_sector']['JSD'].values)
                  for res in ALL_RESULTS if not res['df_jsd_sector'].empty)

    for k, (ax, res) in enumerate(zip(axes, ALL_RESULTS)):
        piv = (res['df_jsd_sector']
               .pivot(index='Sector', columns='Variable', values='JSD')
               .reindex(columns=VAR_LABELS))
        sns.heatmap(piv, ax=ax, cmap='viridis', vmin=0, vmax=jsd_max,
                    annot=True, fmt='.2f', annot_kws={'size': 7},
                    linewidths=0.4, linecolor='#ffffff', mask=piv.isna(),
                    cbar=(k == n_panels - 1),
                    cbar_kws={'label': 'JSD (bits, $n_{bins}=20$)', 'shrink': 0.8})
        ax.set_title(_panel_title(res), fontweight='bold')
        ax.set_xlabel(''); ax.set_ylabel('Sector' if k == 0 else '')
        ax.tick_params(axis='x', rotation=30)

    fig.suptitle('Jensen–Shannon Divergence by sector (test vs reference distributions)',
                 fontweight='bold', y=1.04)
    plt.tight_layout()
    _save(fig, 'fig_jsd_grid.pdf')
    plt.show()


In [ ]:
# ── 13.3  Fig. C — Wilcoxon signed-rank summary (effect sizes, FDR-highlighted) ─
rows = []
for res in ALL_RESULTS:
    if res['df_sig'].empty:
        continue
    d = res['df_sig'].copy()
    d['Panel'] = f"{res['track_name']} | {res['label_test'].split(' (')[0]} vs {res['label_ref'].split(' (')[0]}"
    rows.append(d)
df_wx = pd.concat(rows, ignore_index=True)

with mpl.rc_context(PUB_RC):
    df_wx = df_wx.dropna(subset=['effect_r']).reset_index(drop=True)
    df_wx['label'] = df_wx['Panel'] + '  —  ' + df_wx['Variable']
    y_pos  = np.arange(len(df_wx))[::-1]
    colors = ['#b2182b' if s else '#9e9e9e' for s in df_wx['sig (FDR)']]

    fig, ax = plt.subplots(figsize=(7.5, max(3.5, 0.32 * len(df_wx))))
    ax.barh(y_pos, df_wx['effect_r'], color=colors, edgecolor='white')
    for y, (_, r) in zip(y_pos, df_wx.iterrows()):
        ax.text(r['effect_r'] + (0.03 if r['effect_r'] >= 0 else -0.03), y,
                f"p={r['p-value']:.3f} (q={r['p-adj (FDR)']:.3f})",
                va='center', ha='left' if r['effect_r'] >= 0 else 'right', fontsize=7)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_yticks(y_pos); ax.set_yticklabels(df_wx['label'], fontsize=8)
    ax.set_xlim(-1.15, 1.15)
    ax.set_xlabel(r'Matched-pairs rank-biserial $r$  ($\Delta$PE, Wilcoxon signed-rank, paired by sector)')
    ax.set_title('Wilcoxon signed-rank test on ' + r'$\Delta$PE' + ' — effect sizes\n'
                 '(red = significant after Benjamini–Hochberg FDR, $q<0.05$)',
                 fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    _save(fig, 'fig_wilcoxon_summary.pdf')
    plt.show()


In [ ]:
# ── 13.4  Fig. D — DeltaPE vs Delta sector time (pooled scatter, per channel) ───
MARKERS = ['o', 's', '^', 'D', 'v', 'P']

with mpl.rc_context(PUB_RC):
    fig, axes = plt.subplots(2, 2, figsize=(9, 7.5), sharex=False)
    axes = axes.ravel()

    for ax, var_label in zip(axes, VAR_LABELS):
        xs_all, ys_all = [], []
        for k, res in enumerate(ALL_RESULTS):
            if res['df_pe_sector'].empty or res['df_delta'].empty:
                continue
            piv = res['df_pe_sector'].pivot(index='Sector', columns='Variable', values='DeltaPE')
            if var_label not in piv.columns:
                continue
            merged = (piv[[var_label]]
                      .merge(res['df_delta'].set_index('Sector'),
                             left_index=True, right_index=True)
                      .dropna())
            x, y = merged[var_label].values, merged['DeltaTime_s'].values
            xs_all += list(x); ys_all += list(y)
            short = f"{res['track_name'].split(' ')[0]} | {res['label_test'].split(' (')[0]} vs {res['label_ref'].split(' (')[0]}"
            ax.scatter(x, y, marker=MARKERS[k % len(MARKERS)], s=32, alpha=0.8,
                       edgecolor='black', linewidth=0.4, label=short)

        if len(xs_all) >= 4:
            rho, p_rho = spearmanr(xs_all, ys_all)
            ax.annotate(rf'pooled Spearman $\rho$ = {rho:.2f}  (p = {p_rho:.3f}, n = {len(xs_all)})',
                        xy=(0.03, 0.95), xycoords='axes fraction', va='top', fontsize=8,
                        bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#999999', alpha=0.9))
        ax.axhline(0, color='gray', lw=0.7); ax.axvline(0, color='gray', lw=0.7)
        ax.set_title(var_label, fontweight='bold')
        ax.set_xlabel(r'$\Delta$PE (test $-$ reference)')
        ax.set_ylabel(r'$\Delta$ sector time (s)')
        ax.grid(True, alpha=0.3)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False,
               bbox_to_anchor=(0.5, -0.04))
    fig.suptitle(r'$\Delta$PE vs $\Delta$ sector time — all tracks and skill-tier comparisons',
                 fontweight='bold')
    plt.tight_layout(rect=[0, 0.02, 1, 0.97])
    _save(fig, 'fig_deltape_vs_deltatime.pdf')
    plt.show()
